In [ ]:
!pip install -q \
    google-cloud-bigquery \
    google-cloud-bigquery-storage \
    db-dtypes \
    pandas \
    pyarrow \
    pyspark

In [ ]:
import os
from datetime import datetime
from pathlib import Path
from typing import Dict

import pandas as pd

from google.colab import auth, drive
from google.cloud import bigquery

from pyspark.sql import DataFrame, SparkSession

In [ ]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
auth.authenticate_user()

print("Autenticação no Google Cloud concluída.")

Autenticação no Google Cloud concluída.


In [ ]:
PROJECT_ID = "steady-voltage-502401-p4"

In [ ]:
bq_client = bigquery.Client(
    project=PROJECT_ID
)

spark = (
    SparkSession.builder
    .appName("TechChallenge_Bronze_BigQuery")
    .getOrCreate()
)

print(f"Projeto de execução: {PROJECT_ID}")
print("BigQuery e Spark iniciados.")

Projeto de execução: steady-voltage-502401-p4
BigQuery e Spark iniciados.


In [ ]:
BASE_PATH = "/content/drive/MyDrive/TechChallenge_Fase2"

DATA_PATH = f"{BASE_PATH}/data"
BRONZE_PATH = f"{DATA_PATH}/bronze"
SILVER_PATH = f"{DATA_PATH}/silver"
GOLD_PATH = f"{DATA_PATH}/gold"

OUTPUT_PATH = f"{BASE_PATH}/outputs"
SQL_PATH = f"{BASE_PATH}/sql"

for caminho in [
    BRONZE_PATH,
    SILVER_PATH,
    GOLD_PATH,
    OUTPUT_PATH,
    SQL_PATH
]:
    os.makedirs(caminho, exist_ok=True)

print("Estrutura de diretórios pronta.")

Estrutura de diretórios pronta.


In [ ]:
QUERY_LISTAR_TABELAS = """
SELECT
    table_name,
    table_type
FROM
    `basedosdados.br_inep_avaliacao_alfabetizacao.INFORMATION_SCHEMA.TABLES`
ORDER BY
    table_name
"""

tabelas_disponiveis = bq_client.query(
    QUERY_LISTAR_TABELAS
).to_dataframe()

tabelas_disponiveis

,table_name,table_type
0,alunos,BASE TABLE
1,dicionario,BASE TABLE
2,meta_alfabetizacao_brasil,BASE TABLE
3,meta_alfabetizacao_municipio,BASE TABLE
4,meta_alfabetizacao_uf,BASE TABLE
5,municipio,BASE TABLE
6,uf,BASE TABLE


In [ ]:
CONSULTAS = {
    "municipio": """
        SELECT
            ano,
            id_municipio,
            serie,
            rede,
            taxa_alfabetizacao,
            media_portugues,
            proporcao_aluno_nivel_0,
            proporcao_aluno_nivel_1,
            proporcao_aluno_nivel_2,
            proporcao_aluno_nivel_3,
            proporcao_aluno_nivel_4,
            proporcao_aluno_nivel_5,
            proporcao_aluno_nivel_6,
            proporcao_aluno_nivel_7,
            proporcao_aluno_nivel_8
        FROM
            `basedosdados.br_inep_avaliacao_alfabetizacao.municipio`
        WHERE
            ano >= 2023
    """,

    "uf": """
        SELECT
            ano,
            sigla_uf,
            serie,
            rede,
            taxa_alfabetizacao,
            media_portugues,
            proporcao_aluno_nivel_0,
            proporcao_aluno_nivel_1,
            proporcao_aluno_nivel_2,
            proporcao_aluno_nivel_3,
            proporcao_aluno_nivel_4,
            proporcao_aluno_nivel_5,
            proporcao_aluno_nivel_6,
            proporcao_aluno_nivel_7,
            proporcao_aluno_nivel_8
        FROM
            `basedosdados.br_inep_avaliacao_alfabetizacao.uf`
        WHERE
            ano >= 2023
    """,

    "meta_municipio": """
        SELECT
            ano,
            id_municipio,
            rede,
            taxa_alfabetizacao,
            meta_alfabetizacao_2024,
            meta_alfabetizacao_2025,
            meta_alfabetizacao_2026,
            meta_alfabetizacao_2027,
            meta_alfabetizacao_2028,
            meta_alfabetizacao_2029,
            meta_alfabetizacao_2030,
            nivel_alfabetizacao,
            percentual_participacao
        FROM
            `basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_municipio`
        WHERE
            ano >= 2023
    """,

    "meta_uf": """
        SELECT
            ano,
            sigla_uf,
            rede,
            taxa_alfabetizacao,
            meta_alfabetizacao_2024,
            meta_alfabetizacao_2025,
            meta_alfabetizacao_2026,
            meta_alfabetizacao_2027,
            meta_alfabetizacao_2028,
            meta_alfabetizacao_2029,
            meta_alfabetizacao_2030,
            percentual_participacao
        FROM
            `basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf`
        WHERE
            ano >= 2023
    """,

    "meta_brasil": """
        SELECT
            ano,
            rede,
            taxa_alfabetizacao,
            meta_alfabetizacao_2024,
            meta_alfabetizacao_2025,
            meta_alfabetizacao_2026,
            meta_alfabetizacao_2027,
            meta_alfabetizacao_2028,
            meta_alfabetizacao_2029,
            meta_alfabetizacao_2030,
            percentual_participacao
        FROM
            `basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil`
        WHERE
            ano >= 2023
    """,

    "alunos": """
        SELECT
            ano,
            id_municipio,
            id_escola,

            TO_HEX(
                SHA256(
                    CAST(id_aluno AS STRING)
                )
            ) AS id_aluno_hash,

            caderno,
            serie,
            rede,
            presenca,
            preenchimento_caderno,
            alfabetizado,
            proficiencia,
            peso_aluno

        FROM
            `basedosdados.br_inep_avaliacao_alfabetizacao.alunos`

        WHERE
            ano >= 2023
    """
}

In [ ]:
def estimar_consulta(
    client: bigquery.Client,
    consulta: str
) -> Dict[str, float]:
    """
    Executa um dry run e retorna o volume estimado da consulta.
    """

    job_config = bigquery.QueryJobConfig(
        dry_run=True,
        use_query_cache=False
    )

    job = client.query(
        consulta,
        job_config=job_config
    )

    bytes_processados = job.total_bytes_processed

    return {
        "bytes": float(bytes_processados),
        "mb": bytes_processados / (1024 ** 2),
        "gb": bytes_processados / (1024 ** 3)
    }

In [ ]:
estimativas = []

for nome, consulta in CONSULTAS.items():
    estimativa = estimar_consulta(
        bq_client,
        consulta
    )

    estimativas.append({
        "tabela": nome,
        "mb_estimados": round(estimativa["mb"], 2),
        "gb_estimados": round(estimativa["gb"], 4)
    })

estimativas_pd = pd.DataFrame(estimativas)

estimativas_pd

,tabela,mb_estimados,gb_estimados
0,municipio,1.75,0.0017
1,uf,0.01,0.0000
2,meta_municipio,1.10,0.0011
3,meta_uf,0.01,0.0000
4,meta_brasil,0.00,0.0000
5,alunos,256.10,0.2501


In [ ]:
LIMITE_GB_POR_CONSULTA = 1.0

consultas_acima_limite = estimativas_pd[
    estimativas_pd["gb_estimados"]
    > LIMITE_GB_POR_CONSULTA
]

if not consultas_acima_limite.empty:
    raise RuntimeError(
        "Existem consultas acima do limite de segurança:\n"
        f"{consultas_acima_limite}"
    )

print("Todas as consultas estão dentro do limite configurado.")

Todas as consultas estão dentro do limite configurado.


In [ ]:
def executar_ingestao_batch(
    client: bigquery.Client,
    nome_tabela: str,
    consulta: str
) -> pd.DataFrame:
    """
    Executa uma consulta no BigQuery e adiciona
    metadados de rastreabilidade da ingestão.
    """

    inicio = datetime.now()

    estimativa = estimar_consulta(
        client,
        consulta
    )

    print(
        f"[INFO] {nome_tabela}: "
        f"{estimativa['mb']:.2f} MB estimados."
    )

    dataframe = (
        client.query(consulta)
        .result()
        .to_dataframe()
    )

    dataframe["data_ingestao"] = inicio
    dataframe["origem"] = "Base dos Dados - BigQuery"
    dataframe["modo_ingestao"] = "batch"
    dataframe["projeto_execucao"] = PROJECT_ID

    duracao = (
        datetime.now() - inicio
    ).total_seconds()

    print(
        f"[INFO] {nome_tabela}: "
        f"{len(dataframe):,} registros carregados "
        f"em {duracao:.2f} segundos."
    )

    return dataframe

In [ ]:
TABELAS_MENORES = [
    "municipio",
    "uf",
    "meta_municipio",
    "meta_uf",
    "meta_brasil"
]

datasets_pandas = {}

for nome in TABELAS_MENORES:
    datasets_pandas[nome] = executar_ingestao_batch(
        client=bq_client,
        nome_tabela=nome,
        consulta=CONSULTAS[nome]
    )

[INFO] municipio: 1.75 MB estimados.
[INFO] municipio: 23,995 registros carregados em 2.89 segundos.
[INFO] uf: 0.01 MB estimados.
[INFO] uf: 145 registros carregados em 2.99 segundos.
[INFO] meta_municipio: 1.10 MB estimados.
[INFO] meta_municipio: 10,704 registros carregados em 2.49 segundos.
[INFO] meta_uf: 0.01 MB estimados.
[INFO] meta_uf: 81 registros carregados em 2.77 segundos.
[INFO] meta_brasil: 0.00 MB estimados.
[INFO] meta_brasil: 3 registros carregados em 2.42 segundos.


In [ ]:
for nome, dataframe in datasets_pandas.items():
    print(
        f"{nome}: "
        f"{dataframe.shape[0]:,} linhas e "
        f"{dataframe.shape[1]} colunas"
    )

municipio: 23,995 linhas e 19 colunas
uf: 145 linhas e 19 colunas
meta_municipio: 10,704 linhas e 17 colunas
meta_uf: 81 linhas e 16 colunas
meta_brasil: 3 linhas e 15 colunas


In [ ]:
datasets_spark = {}

for nome, dataframe in datasets_pandas.items():
    datasets_spark[nome] = spark.createDataFrame(
        dataframe
    )

print("DataFrames convertidos para Spark.")

DataFrames convertidos para Spark.


In [ ]:
for nome, dataframe in datasets_spark.items():
    caminho = f"{BRONZE_PATH}/{nome}"

    (
        dataframe.write
        .mode("overwrite")
        .partitionBy("ano")
        .parquet(caminho)
    )

    print(f"Bronze gravada: {nome}")

Bronze gravada: municipio
Bronze gravada: uf
Bronze gravada: meta_municipio
Bronze gravada: meta_uf
Bronze gravada: meta_brasil


In [ ]:
QUERY_ALUNOS_AGREGADOS = """
SELECT
    ano,
    id_municipio,
    id_escola,
    serie,
    rede,

    COUNT(*) AS total_registros,

    COUNTIF(
        LOWER(TRIM(presenca)) IN ('1', 'sim', 's', 'true')
    ) AS total_presentes,

    COUNTIF(
        LOWER(TRIM(alfabetizado)) IN ('1', 'sim', 's', 'true')
    ) AS total_alfabetizados,

    COUNTIF(
        LOWER(TRIM(preenchimento_caderno))
        IN ('1', 'sim', 's', 'true')
    ) AS total_cadernos_preenchidos,

    AVG(proficiencia) AS proficiencia_media,

    AVG(peso_aluno) AS peso_medio,

    SAFE_MULTIPLY(
        SAFE_DIVIDE(
            COUNTIF(
                LOWER(TRIM(presenca))
                IN ('1', 'sim', 's', 'true')
            ),
            COUNT(*)
        ),
        100
    ) AS taxa_presenca,

    SAFE_MULTIPLY(
        SAFE_DIVIDE(
            COUNTIF(
                LOWER(TRIM(alfabetizado))
                IN ('1', 'sim', 's', 'true')
            ),
            COUNT(*)
        ),
        100
    ) AS taxa_alfabetizacao_alunos

FROM
    `basedosdados.br_inep_avaliacao_alfabetizacao.alunos`

WHERE
    ano >= 2023

GROUP BY
    ano,
    id_municipio,
    id_escola,
    serie,
    rede
"""

In [ ]:
QUERY_DOMINIOS_ALUNOS = """
SELECT
    'presenca' AS campo,
    presenca AS valor,
    COUNT(*) AS quantidade
FROM
    `basedosdados.br_inep_avaliacao_alfabetizacao.alunos`
GROUP BY presenca

UNION ALL

SELECT
    'alfabetizado' AS campo,
    alfabetizado AS valor,
    COUNT(*) AS quantidade
FROM
    `basedosdados.br_inep_avaliacao_alfabetizacao.alunos`
GROUP BY alfabetizado

UNION ALL

SELECT
    'preenchimento_caderno' AS campo,
    preenchimento_caderno AS valor,
    COUNT(*) AS quantidade
FROM
    `basedosdados.br_inep_avaliacao_alfabetizacao.alunos`
GROUP BY preenchimento_caderno

ORDER BY campo, quantidade DESC
"""

dominios_alunos = bq_client.query(
    QUERY_DOMINIOS_ALUNOS
).to_dataframe()

dominios_alunos

,campo,valor,quantidade
0,alfabetizado,1,1984546
1,alfabetizado,0,1883453
2,preenchimento_caderno,1,3354661
3,preenchimento_caderno,0,513338
4,presenca,1,3355846
5,presenca,0,512153


In [ ]:
estimativa_alunos = estimar_consulta(
    bq_client,
    QUERY_ALUNOS_AGREGADOS
)

print(
    f"Consulta alunos agregados: "
    f"{estimativa_alunos['mb']:.2f} MB "
    f"({estimativa_alunos['gb']:.4f} GB)"
)

Consulta alunos agregados: 206.12 MB (0.2013 GB)


In [ ]:
from datetime import datetime

print("Iniciando agregação da tabela alunos no BigQuery...")

alunos_agregados_pd = (
    bq_client
    .query(QUERY_ALUNOS_AGREGADOS)
    .result()
    .to_dataframe()
)

alunos_agregados_pd["data_ingestao"] = datetime.now()
alunos_agregados_pd["origem"] = "Base dos Dados - BigQuery"
alunos_agregados_pd["modo_ingestao"] = "batch"
alunos_agregados_pd["granularidade"] = (
    "ano_municipio_escola_serie_rede"
)

print(
    f" {len(alunos_agregados_pd):,} "
    "registros agregados carregados."
)

Iniciando agregação da tabela alunos no BigQuery...
 79,273 registros agregados carregados.


In [ ]:
memoria_mb = (
    alunos_agregados_pd
    .memory_usage(deep=True)
    .sum()
    / 1024**2
)

print(f"Memória ocupada no Pandas: {memoria_mb:.2f} MB")

Memória ocupada no Pandas: 38.25 MB


In [ ]:
alunos_agregados_spark = spark.createDataFrame(
    alunos_agregados_pd
)

alunos_agregados_spark.printSchema()

root
 |-- ano: long (nullable = true)
 |-- id_municipio: string (nullable = true)
 |-- id_escola: string (nullable = true)
 |-- serie: string (nullable = true)
 |-- rede: string (nullable = true)
 |-- total_registros: long (nullable = true)
 |-- total_presentes: long (nullable = true)
 |-- total_alfabetizados: long (nullable = true)
 |-- total_cadernos_preenchidos: long (nullable = true)
 |-- proficiencia_media: double (nullable = true)
 |-- peso_medio: double (nullable = true)
 |-- taxa_presenca: double (nullable = true)
 |-- taxa_alfabetizacao_alunos: double (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- origem: string (nullable = true)
 |-- modo_ingestao: string (nullable = true)
 |-- granularidade: string (nullable = true)



In [ ]:
CAMINHO_BRONZE_ALUNOS = (
    f"{BRONZE_PATH}/alunos_agregados"
)

(
    alunos_agregados_spark
    .write
    .mode("overwrite")
    .partitionBy("ano")
    .parquet(CAMINHO_BRONZE_ALUNOS)
)

print(
    "Bronze agregada de alunos salva com particionamento por ano."
)

Bronze agregada de alunos salva com particionamento por ano.


In [ ]:
alunos_bronze = spark.read.parquet(
    CAMINHO_BRONZE_ALUNOS
)

print("Registros:", alunos_bronze.count())
print("Colunas:", len(alunos_bronze.columns))

alunos_bronze.show(
    10,
    truncate=False
)

Registros: 79273
Colunas: 17
+------------+---------+-----+----+---------------+---------------+-------------------+--------------------------+------------------+------------------+-----------------+-------------------------+--------------------------+-------------------------+-------------+-------------------------------+----+
|id_municipio|id_escola|serie|rede|total_registros|total_presentes|total_alfabetizados|total_cadernos_preenchidos|proficiencia_media|peso_medio        |taxa_presenca    |taxa_alfabetizacao_alunos|data_ingestao             |origem                   |modo_ingestao|granularidade                  |ano |
+------------+---------+-----+----+---------------+---------------+-------------------+--------------------------+------------------+------------------+-----------------+-------------------------+--------------------------+-------------------------+-------------+-------------------------------+----+
|2304400     |60008051 |2    |3   |77             |74             |6

In [ ]:
alunos_pd = executar_ingestao_batch(
    client=bq_client,
    nome_tabela="alunos",
    consulta=CONSULTAS["alunos"]
)

[INFO] alunos: 256.10 MB estimados.
[INFO] alunos: 3,867,999 registros carregados em 17.90 segundos.


In [ ]:
assert "id_aluno" not in alunos_pd.columns
assert "id_aluno_hash" in alunos_pd.columns

print("Identificador original não está presente.")
print("Identificador pseudonimizado disponível.")

Identificador original não está presente.
Identificador pseudonimizado disponível.


In [ ]:
alunos_spark = spark.createDataFrame(
    alunos_pd
)

(
    alunos_spark.write
    .mode("overwrite")
    .partitionBy("ano")
    .parquet(
        f"{BRONZE_PATH}/alunos"
    )
)

print("Bronze de alunos gravada.")

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.readRDDFromFile.
: java.lang.OutOfMemoryError: Java heap space


In [ ]:
TABELAS_BRONZE = [
    "municipio",
    "uf",
    "meta_municipio",
    "meta_uf",
    "meta_brasil",
    "alunos"
]

relatorio_bronze = []

for nome in TABELAS_BRONZE:
    caminho = f"{BRONZE_PATH}/{nome}"

    dataframe = spark.read.parquet(
        caminho
    )

    relatorio_bronze.append({
        "tabela": nome,
        "registros": dataframe.count(),
        "colunas": len(dataframe.columns),
        "possui_ano": "ano" in dataframe.columns,
        "possui_origem": "origem" in dataframe.columns,
        "possui_data_ingestao": (
            "data_ingestao" in dataframe.columns
        )
    })

relatorio_bronze_pd = pd.DataFrame(
    relatorio_bronze
)

relatorio_bronze_pd

In [ ]:
estimativas_pd.to_csv(
    f"{OUTPUT_PATH}/estimativas_bigquery.csv",
    index=False
)

relatorio_bronze_pd.to_csv(
    f"{OUTPUT_PATH}/relatorio_ingestao_bronze.csv",
    index=False
)

print("Relatórios da ingestão exportados.")